# Modelado — ablación de vecinos y jerarquía

Este notebook llama a `src.modeling.ablacion_espacial`. Compara la base de seis variables con los bloques de vecinos, jerarquía y ambos sobre las mismas filas. Mantiene F1 como criterio provisional para el umbral de clasificación y para seleccionar variables con las temporadas 2021–2023. Los bloques de prueba 2024 y 2025 se evalúan después de esa selección.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modeling.ablacion_espacial import ejecutar_ablacion_espacial

reporte = ejecutar_ablacion_espacial()

In [ ]:
import pandas as pd

filas = []
for h, bloque in reporte["horizontes"].items():
    elegidas = bloque["elegidas_solo_validacion"]
    for variante, resultado in bloque["variantes"].items():
        folds = resultado["folds_validacion"]
        filas.append({"h": int(h), "variante": variante, "n_variables": len(resultado["columnas"]),
                      "F1_regresion_validacion": sum(f["regresion"]["alerta_derivada"]["f1"] for f in folds) / len(folds),
                      "F1_clasificacion_validacion": sum(f["clasificacion"]["alerta_directa"]["f1"] for f in folds) / len(folds),
                      "elegida_regresion": variante == elegidas["regresion"],
                      "elegida_clasificacion": variante == elegidas["clasificacion"]})
display(pd.DataFrame(filas).sort_values(["h", "variante"]))

In [ ]:
filas = []
for h, bloque in reporte["horizontes"].items():
    for variante, resultado in bloque["variantes"].items():
        for fold in resultado["folds_prueba"]:
            for enfoque, metrica in (("regresion", fold["regresion"]["alerta_derivada"]),
                                     ("clasificacion", fold["clasificacion"]["alerta_directa"])):
                filas.append({"h": int(h), "bloque": fold["bloque"], "variante": variante,
                              "enfoque": enfoque, "positivos": metrica["positivos"],
                              "VP": metrica["vp"], "FP": metrica["fp"],
                              "F1": metrica["f1"]})
display(pd.DataFrame(filas).sort_values(["h", "bloque", "enfoque", "variante"]))

Los bloques espaciales fueron seleccionados únicamente por F1 de validación. La prueba principal de 2024 ya se había inspeccionado en el experimento base; sus resultados aquí sirven como contraste de la señal nueva, no como un test completamente virgen. El informe se encuentra en `docs/modeling/ablacion_espacial.md`. Clima y demografía se revisarán en fases siguientes.